In [ ]:
# tuned_labse_training_single_run.py
# ================== CLEAN SETUP ==================
import os, warnings, logging, re, random
os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ================== IMPORTS ==================
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import nltk
from nltk.corpus import wordnet
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from torch.nn import CrossEntropyLoss

nltk.download("wordnet")

# ================== TEXT CLEANING ==================
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

# ================== AUGMENTATION (ENHANCED) ==================
def synonym_replacement(sentence, n=1):
    words = sentence.split()
    new_words = words.copy()
    candidates = [w for w in words if len(w) > 1 and len(wordnet.synsets(w)) > 0]
    random.shuffle(candidates)
    num_replaced = 0
    for w in candidates:
        synonyms = wordnet.synsets(w)
        synonym_words = [lemma.name().replace("_", " ") for syn in synonyms for lemma in syn.lemmas() if lemma.name() != w]
        if synonym_words:
            synonym = random.choice(synonym_words)
            new_words = [synonym if token == w else token for token in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return " ".join(new_words)

def random_deletion(sentence, p=0.1):
    words = sentence.split()
    if len(words) == 1:
        return sentence
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        return random.choice(words)
    return " ".join(new_words)

def random_swap(sentence, n_swaps=1):
    words = sentence.split()
    if len(words) < 2:
        return sentence
    for _ in range(n_swaps):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return " ".join(words)

def random_insertion(sentence, n_insert=1):
    words = sentence.split()
    for _ in range(n_insert):
        candidates = [w for w in words if len(wordnet.synsets(w)) > 0]
        if not candidates:
            break
        word_to_replace = random.choice(candidates)
        synonyms = wordnet.synsets(word_to_replace)
        synonym_words = [lemma.name().replace("_", " ") for syn in synonyms for lemma in syn.lemmas() if lemma.name() != word_to_replace]
        if synonym_words:
            insert_word = random.choice(synonym_words)
            pos = random.randrange(len(words) + 1)
            words.insert(pos, insert_word)
    return " ".join(words)

def augment_text(text):
    p = random.random()
    if p < 0.35:
        return synonym_replacement(text, n=1)
    elif p < 0.6:
        return random_deletion(text, p=0.15)
    elif p < 0.8:
        return random_swap(text, n_swaps=1)
    else:
        return random_insertion(text, n_insert=1)

# ================== LOAD DATA ==================
df = pd.read_csv("Dataset_1_2_3_4.csv")

if df["label"].dtype == "object":
    label_mapping = {"Non-Hate": 0, "Hate": 1}
    df["label"] = df["label"].map(label_mapping)

df["label"] = df["label"].astype(int)
df["text"] = df["text"].apply(clean_text)

# ================== STRONGER AUGMENTATION ==================
AUGMENT_FACTOR = 2
AUGMENT_PROB = 0.4

augmented_rows = []
for text, label in zip(df["text"], df["label"]):
    if random.random() < AUGMENT_PROB:
        for _ in range(AUGMENT_FACTOR):
            augmented_text = augment_text(text)
            augmented_rows.append({"text": augmented_text, "label": label})

df_aug = pd.DataFrame(augmented_rows)
if not df_aug.empty:
    df = pd.concat([df, df_aug]).reset_index(drop=True)

df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

# ================== SPLIT DATASET ==================
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ================== CLASS WEIGHTS ==================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ================== TOKENIZER / MODEL ==================
MODEL_NAME = "sentence-transformers/LaBSE"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ================== CUSTOM DATASET ==================
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=tokenizer, max_len=192):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ================== CUSTOM CLASSIFIER ==================
class CustomSequenceClassifier(nn.Module):
    def __init__(self, base_model_name, num_labels=2, hidden_dropout_prob=0.2, intermediate_dim=None, activation="gelu"):
        super().__init__()
        self.base = AutoModel.from_pretrained(base_model_name)
        base_hidden = self.base.config.hidden_size
        if intermediate_dim is None:
            intermediate_dim = base_hidden // 2
        self.dropout = nn.Dropout(hidden_dropout_prob)
        self.dense = nn.Linear(base_hidden, intermediate_dim)
        if activation == "gelu":
            self.act = nn.GELU()
        elif activation == "relu":
            self.act = nn.ReLU()
        elif activation == "leaky":
            self.act = nn.LeakyReLU()
        else:
            self.act = nn.GELU()
        self.out_proj = nn.Linear(intermediate_dim, num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = getattr(outputs, "pooler_output", None)
        if pooled is None:
            last_hidden = outputs.last_hidden_state
            pooled = last_hidden.mean(dim=1)
        x = self.dropout(pooled)
        x = self.dense(x)
        x = self.act(x)
        logits = self.out_proj(x)
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

# ================== PREPARE DATASETS ==================
train_dataset = TextDataset(train_texts, train_labels)
test_dataset  = TextDataset(test_texts, test_labels)

# ================== DEVICE INFO ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================== TRAINING (Single Run, 10 Epochs) ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

model = CustomSequenceClassifier(MODEL_NAME, num_labels=2, hidden_dropout_prob=0.2)
model.to(device)

training_args = TrainingArguments(
    output_dir="./results_labse_single_run",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,  # <<--- only 10 epochs
    weight_decay=0.01,
    warmup_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    seed=42,
    dataloader_drop_last=False,
    report_to="none"
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"] if isinstance(outputs, dict) and "loss" in outputs else outputs[0]
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # remove if you want fixed 10 epochs
)

trainer.train()

# ================== FINAL EVALUATION ==================
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=-1)

print("\n✅ Test Accuracy:", accuracy_score(test_labels, y_pred))
print("\n✅ Classification Report:\n", classification_report(test_labels, y_pred, target_names=["Non-Hate", "Hate"]))


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Using device: cuda
{'loss': 0.4887, 'grad_norm': 13.186166763305664, 'learning_rate': 1.8238532110091744e-05, 'epoch': 1.0}
{'eval_loss': 0.34932664036750793, 'eval_accuracy': 0.8592233009708737, 'eval_runtime': 8.5342, 'eval_samples_per_second': 362.071, 'eval_steps_per_second': 22.732, 'epoch': 1.0}
{'loss': 0.2506, 'grad_norm': 14.949686050415039, 'learning_rate': 1.6212319790301444e-05, 'epoch': 2.0}
{'eval_loss': 0.3123088479042053, 'eval_accuracy': 0.9019417475728155, 'eval_runtime': 8.6034, 'eval_samples_per_second': 359.159, 'eval_steps_per_second': 22.549, 'epoch': 2.0}
{'loss': 0.1146, 'grad_norm': 0.04493790492415428, 'learning_rate': 1.4186107470511141e-05, 'epoch': 3.0}
{'eval_loss': 0.3766459822654724, 'eval_accuracy': 0.9032362459546925, 'eval_runtime': 8.6092, 'eval_samples_per_second': 358.918, 'eval_steps_per_second': 22.534, 'epoch': 3.0}
{'loss': 0.0604, 'grad_norm': inf, 'learning_rate': 1.215989515072084e-05, 'epoch': 4.0}
{'eval_loss': 0.4553946256637573, 'eval_a